In [1]:
%pip install folium
%pip install wget
%pip install pandas

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import folium
import wget
import pandas as pd

In [3]:
from folium.plugins import MarkerCluster
from folium.plugins import MousePosition
from folium.features import DivIcon

In [4]:
spacex_csv_file = wget.download('https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/spacex_launch_geo.csv')
spacex_df = pd.read_csv(spacex_csv_file)

In [5]:
spacex_df = spacex_df[['Launch Site', 'Lat', 'Long', 'class']]
launch_sites_df = spacex_df.groupby(['Launch Site'], as_index=False).first()
launch_sites_df = launch_sites_df[['Launch Site', 'Lat', 'Long']]
launch_sites_df

,Launch Site,Lat,Long
0,CCAFS LC-40,28.562302,-80.577356
1,CCAFS SLC-40,28.563197,-80.576820
2,KSC LC-39A,28.573255,-80.646895
3,VAFB SLC-4E,34.632834,-120.610745


In [6]:
nasa_coordinate = [29.559684888503615, -95.0830971930759]
site_map = folium.Map(location=nasa_coordinate, zoom_start=10)

In [7]:
circle = folium.Circle(
    nasa_coordinate, 
    radius=1000, color='#d35400', 
    fill=True).add_child(folium.Popup('NASA Johnson Space Center'))

marker = folium.map.Marker(
    nasa_coordinate,
    icon = DivIcon(
       icon_size=(20,20),
       icon_anchor=(0,0),
       html='<div style="font-size: 12; color:#d35400;"><b>%s</b></div>' % 'NASA JSC' 
    )
)

site_map.add_child(circle)
site_map.add_child(marker)

In [8]:
site_map = folium.Map(location=nasa_coordinate, zoom_start=5)

for index, row in launch_sites_df.iterrows():
    coordinate = [row['Lat'], row['Long']]
    launch_site = row['Launch Site']

    circle = folium.Circle(
        coordinate,
        radius=1000,
        color='#000000',
        fill=True
    ).add_child(folium.Popup(launch_site))

    marker = folium.Marker(
        coordinate,
        icon=DivIcon(
            icon_size=(20, 20),
            icon_anchor=(0, 0),
            html='<div style="font-size: 12; color:#d35400;"><b>%s</b></div>' % launch_site
        )
    )

    site_map.add_child(circle)
    site_map.add_child(marker)

site_map

In [9]:
spacex_df.tail(10)

,Launch Site,Lat,Long,class
46,KSC LC-39A,28.573255,-80.646895,1
47,KSC LC-39A,28.573255,-80.646895,1
48,KSC LC-39A,28.573255,-80.646895,1
49,CCAFS SLC-40,28.563197,-80.576820,1
50,CCAFS SLC-40,28.563197,-80.576820,1
51,CCAFS SLC-40,28.563197,-80.576820,0
52,CCAFS SLC-40,28.563197,-80.576820,0
53,CCAFS SLC-40,28.563197,-80.576820,0
54,CCAFS SLC-40,28.563197,-80.576820,1
55,CCAFS SLC-40,28.563197,-80.576820,0


In [10]:
marker_cluster = MarkerCluster()

In [11]:
def assign_marker_color(launch_outcome):
    if launch_outcome == 1:
        return 'green'
    else:
        return 'red'

spacex_df['marker_color'] = spacex_df['class'].apply(assign_marker_color)
spacex_df.tail(10)

,Launch Site,Lat,Long,class,marker_color
46,KSC LC-39A,28.573255,-80.646895,1,green
47,KSC LC-39A,28.573255,-80.646895,1,green
48,KSC LC-39A,28.573255,-80.646895,1,green
49,CCAFS SLC-40,28.563197,-80.576820,1,green
50,CCAFS SLC-40,28.563197,-80.576820,1,green
51,CCAFS SLC-40,28.563197,-80.576820,0,red
52,CCAFS SLC-40,28.563197,-80.576820,0,red
53,CCAFS SLC-40,28.563197,-80.576820,0,red
54,CCAFS SLC-40,28.563197,-80.576820,1,green
55,CCAFS SLC-40,28.563197,-80.576820,0,red


In [12]:
site_map.add_child(marker_cluster)

for index, record in spacex_df.iterrows():
    marker = folium.Marker(
        location = [record['Lat'], record['Long']],
        icon = folium.Icon(
            color = 'white',
            icon_color = record['marker_color']
        )
    )

    marker_cluster.add_child(marker)

site_map

In [14]:
formatter = "function(num) {return L.Util.formatNum(num, 5);};"
mouse_position = MousePosition(
    position = 'topright',
    separator = 'Long: ',
    empty_string = 'NaN',
    lng_first = False,
    prefix = 'Lat: ',
    lat_formatter = formatter,
    lng_formatter = formatter,
)

site_map.add_child(mouse_position)
site_map

In [15]:
from math import sin, cos, sqrt, atan2, radians

def calculate_distance(lat1, lon1, lat2, lon2):
    R = 6373.0

    lat1 = radians(lat1)
    lon1 = radians(lon1)
    lat2 = radians(lat2)
    lon2 = radians(lon2)

    dlon = lon2 - lon1
    dlat = lat2 - lat1

    a = sin(dlat/2)**2 + cos(lat1) * cos(lat2) * sin(dlon/2)**2
    c = 2 * atan2(sqrt(a), sqrt(1 - a))

    distance = R*c
    return distance

In [16]:
launch_site_lat = 28.562302
launch_site_lon = -80.577356

coastline_lat = 28.56367
coastline_lon = -80.57163

distance_coastline = calculate_distance(
    launch_site_lat, launch_site_lon, coastline_lat,coastline_lon
)

print("Distance to coastline:", round(distance_coastline, 2), "KM")

Distance to coastline: 0.58 KM


In [18]:
coordinate = [coastline_lat, coastline_lon]

distance_marker = folium.Marker(
    coordinate,
    icon = DivIcon(
        icon_size = (20, 20),
        icon_anchor = (0, 0),
        html = '<div style="font-size: 12; color:#d35400;"><b>%s</b></div>' % "{:10.2f} KM".format(distance_coastline),
    )
)

site_map.add_child(distance_marker)
site_map

In [20]:
coordinates = [
    [launch_site_lat, launch_site_lon],
    [coastline_lat, coastline_lon]
]

lines = folium.PolyLine(
    locations = coordinates,
    weight = 1
)

site_map.add_child(lines)
site_map

In [21]:
launch_site_lat = 28.562302
launch_site_lon = -80.577356

railway_lat = 28.5720
railway_lon = -80.5850

distance_railway = calculate_distance(
    launch_site_lat, launch_site_lon, railway_lat, railway_lon
)

print("Distance to railway:", round(distance_railway, 2), "KM")

Distance to railway: 1.31 KM


In [22]:
railway_coordinate = [railway_lat, railway_lon]

railway_marker = folium.Marker(
    railway_coordinate,
    icon=DivIcon(
        icon_size=(20, 20),
        icon_anchor=(0, 0),
        html='<div style="font-size: 12; color:#d35400;"><b>%s</b></div>' % "{:.2f} KM".format(distance_railway)
    )
)

site_map.add_child(railway_marker)
site_map

In [23]:
railway_coordinates = [
    [launch_site_lat, launch_site_lon],
    [railway_lat, railway_lon]
]

railway_line = folium.PolyLine(
    locations=railway_coordinates,
    weight=2,
    color='blue'
)

site_map.add_child(railway_line)
site_map

In [24]:
launch_site_lat = 28.562302
launch_site_lon = -80.577356

highway_lat = 28.5630
highway_lon = -80.6050

# Calculate distance to the highway
distance_highway = calculate_distance(
    launch_site_lat,
    launch_site_lon,
    highway_lat,
    highway_lon
)

print("Distance to highway:", round(distance_highway, 2), "KM")

Distance to highway: 2.7 KM


In [25]:
highway_coordinate = [highway_lat, highway_lon]

highway_marker = folium.Marker(
    highway_coordinate,
    icon=DivIcon(
        icon_size=(20, 20),
        icon_anchor=(0, 0),
        html='<div style="font-size: 12; color:#d35400;"><b>%s</b></div>' % "{:.2f} KM".format(distance_highway)
    )
)

site_map.add_child(highway_marker)
site_map

In [26]:
highway_coordinates = [
    [launch_site_lat, launch_site_lon],
    [highway_lat, highway_lon]
]

highway_line = folium.PolyLine(
    locations=highway_coordinates,
    weight=2,
    color='purple'
)

site_map.add_child(highway_line)
site_map

In [27]:
city_lat = 28.3922
city_lon = -80.6077

distance_city = calculate_distance(
    launch_site_lat,
    launch_site_lon,
    city_lat,
    city_lon
)

print("Distance to city:", round(distance_city, 2), "KM")

Distance to city: 19.15 KM


In [28]:
city_coordinate = [city_lat, city_lon]

city_marker = folium.Marker(
    city_coordinate,
    icon=DivIcon(
        icon_size=(20, 20),
        icon_anchor=(0, 0),
        html='<div style="font-size: 12; color:#d35400;"><b>%s</b></div>' % "{:.2f} KM".format(distance_city)
    )
)

site_map.add_child(city_marker)
site_map

In [29]:
city_coordinates = [
    [launch_site_lat, launch_site_lon],
    [city_lat, city_lon]
]

city_line = folium.PolyLine(
    locations=city_coordinates,
    weight=2,
    color='green'
)

site_map.add_child(city_line)
site_map